# Công cụ Tách Vocal Thư viện Nhạc (Meta HTDemucs v4) cho QBH V6.2
Notebook này dùng mô hình **Meta HTDemucs v4** để tách lấy phần giọng hát mộc (ocals.wav) cho các bài hát trong thư viện nhạc (Gallery) của đề tài Query by Humming.
- **Mục đích**: Loại bỏ hoàn toàn guitar điện, trống bass và nhạc cụ đệm, giúp tiếng humming mộc của người dùng đối sánh trực tiếp với giọng ca sĩ chính.
- **Đặc biệt**: Hỗ trợ 3 chế độ (Chạy thử 4 bài khó trong ~2 phút, chạy 52 bài test trong ~25 phút, hoặc toàn bộ 348 bài).
- **GPU**: Tesla T4 trên Google Colab.


In [ ]:
# 1. KIEM TRA GPU T4 VA CAI DAT THU VIEN DEMUCS
import torch, os, subprocess

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type != 'cuda':
    raise RuntimeError('[ERROR] CUDA chua bat! Vao Runtime -> Change runtime type -> Chon T4 GPU -> Save.')

print(f'[OK] GPU Hardware: {torch.cuda.get_device_name(0)}')
print('[INFO] Dang cai dat demucs v4 tu PyPI...')
subprocess.run(['pip', 'install', '-q', 'demucs'], check=True)
print('[OK] Cai dat demucs thanh cong!')


In [ ]:
# 2. KET NOI GOOGLE DRIVE
from google.colab import drive
from pathlib import Path
import shutil, zipfile, json

drive.mount('/content/drive')
print('[OK] Google Drive da ket noi thanh cong!')


In [ ]:
# 3. CHON CHE DO TACH VOCAL VA CAU HINH
# Cac che do:
#   '4_hard_songs'  : Chi tach 4 bai kho (Highway to Hell, Back To Black, No Brainer, Just A Dream) -> ~2 phut
#   'test_split_52' : Tach toan bo 52 bai hat thuoc tap Test -> ~25 phut
#   'all_348'       : Tach toan bo 348 bai trong thu vien Gallery -> ~2.5 gio

SEPARATION_MODE = '4_hard_songs'  # <-- BAN CO THE DOI SANG 'test_split_52' HOAC 'all_348'

WORKSPACE_DIR = Path('/content')
OUTPUT_VOCALS_DIR = Path('/content/qbh_gallery_vocals_v6_2')
OUTPUT_VOCALS_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT_ZIP = Path('/content/drive/MyDrive/qbh_gallery_vocals_v6_2.zip')

print(f'[INFO] Che do tach vocal duoc chon: {SEPARATION_MODE}')


In [ ]:
# 4. DINH VI DATASET VA THU VIEN 2_MUSIC_DATABASE
dataset_zip_cand = [
    Path('/content/drive/MyDrive/datasets_qbh_clean_mirqbsh_70_15_15_v6_1.zip'),
    Path('/content/datasets_qbh_clean_mirqbsh_70_15_15_v6_1.zip')
]
dataset_zip = None
for c in dataset_zip_cand:
    if c.is_file():
        dataset_zip = c
        break

if dataset_zip is None:
    raise FileNotFoundError('[ERROR] Khong tim thay datasets_qbh_clean_mirqbsh_70_15_15_v6_1.zip tren Google Drive.')

print(f'[OK] Tim thay Dataset ZIP: {dataset_zip}')
unpacked_dir = Path('/content/dataset_unpacked')
music_db_dir = None

# Kiem tra neu da giai nen
for cand in unpacked_dir.rglob('2_music_database'):
    if cand.is_dir():
        music_db_dir = cand
        break

if music_db_dir is None:
    print('[INFO] Dang giai nen thu vien 2_music_database tu file zip...')
    with zipfile.ZipFile(dataset_zip, 'r') as z:
        for m in z.namelist():
            if '2_music_database' in m or 'song_name_mapping' in m or 'split_manifest' in m or 'mir_qbsh_extension' in m:
                z.extract(m, unpacked_dir)
    for cand in unpacked_dir.rglob('2_music_database'):
        if cand.is_dir():
            music_db_dir = cand
            break

print(f'[OK] Thu muc 2_music_database: {music_db_dir}')
mapping_file = next(unpacked_dir.rglob('song_name_mapping.json'))
song_mapping = json.loads(mapping_file.read_text(encoding='utf-8'))
manifest_file = next(unpacked_dir.rglob('split_manifest.json'))
split_manifest = json.loads(manifest_file.read_text(encoding='utf-8'))
print(f'[OK] Da nap {len(song_mapping)} mappings va split manifest.')


In [ ]:
# 5. XAC DINH DANH SACH BAI HAT CAN TACH VOCAL
songs_to_process = [] # list of (song_id, song_title, audio_path)

# 4 bai kho dac thu
HARD_SONG_IDS = {
    'song_0009': 'ACDC - Highway to Hell (lyrics)',
    'song_0028': 'Amy Winehouse - Back To Black',
    'song_0096': 'DJ Khaled - No Brainer ft. Justin Bieber, Chance the Rapper, Quavo',
    'song_0219': 'Nelly - Just A Dream'
}

# 45 bai test goc
test_source_titles = {folder for folder, d in split_manifest.items() if d.get('split') == 'test'}
id_to_title = {k: v for k, v in song_mapping.items()}

if SEPARATION_MODE == '4_hard_songs':
    for sid, title in HARD_SONG_IDS.items():
        audio_p = music_db_dir / f'{sid}.mp3'
        if audio_p.is_file():
            songs_to_process.append((sid, title, audio_p))
elif SEPARATION_MODE == 'test_split_52':
    for sid, title in song_mapping.items():
        if title in test_source_titles or sid in HARD_SONG_IDS:
            audio_p = music_db_dir / f'{sid}.mp3'
            if audio_p.is_file():
                songs_to_process.append((sid, title, audio_p))
else: # 'all_348'
    for sid, title in song_mapping.items():
        audio_p = music_db_dir / f'{sid}.mp3'
        if audio_p.is_file():
            songs_to_process.append((sid, title, audio_p))

print(f'[INFO] Tong so bai hat se tach vocal trong dot nay: {len(songs_to_process)} bai')
for s in songs_to_process[:5]:
    print(f'   - {s[0]}: {s[1]}')


In [ ]:
# 6. THUC THI TACH VOCAL BANG DEMUCS V4 (HTDEMUCS)
from tqdm.auto import tqdm
import time

print(f'[START] Bat dau tach vocal voi Demucs v4 tren GPU: {torch.cuda.get_device_name(0)}')
demucs_raw_out = Path('/content/demucs_raw')
demucs_raw_out.mkdir(parents=True, exist_ok=True)

success_count = 0
start_time = time.time()

for sid, title, audio_path in tqdm(songs_to_process, desc='Demucs Separating'):
    target_vocal_file = OUTPUT_VOCALS_DIR / f'{sid}_vocals.wav'
    if target_vocal_file.is_file() and target_vocal_file.stat().st_size > 10000:
        success_count += 1
        continue
    
    # Chay demucs lenh hai stem: chi xuat vocal va no_vocal
    cmd = [
        'demucs',
        '--two-stems=vocals',
        '-n', 'htdemucs',
        '--device', 'cuda',
        '-o', str(demucs_raw_out),
        str(audio_path)
    ]
    res = subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
    if res.returncode != 0:
        print(f'[WARNING] Loi tach bai {sid}: {res.stderr[:200]}')
        continue
    
    # Demucs xuat vao: demucs_raw_out/htdemucs/{stem_name}/vocals.wav
    expected_vocal = demucs_raw_out / 'htdemucs' / audio_path.stem / 'vocals.wav'
    if expected_vocal.is_file():
        shutil.copyfile(expected_vocal, target_vocal_file)
        success_count += 1
    else:
        print(f'[WARNING] Khong tim thay output vocal cho {sid}')

elapsed = round(time.time() - start_time, 1)
print(f'[HOAN THANH] Da tach thanh cong {success_count}/{len(songs_to_process)} bai hat trong {elapsed}s!')


In [ ]:
# 7. DONG GOI VA SAO LUU VAO GOOGLE DRIVE
zip_local = Path('/content/qbh_gallery_vocals_v6_2.zip')
print(f'[INFO] Dang nen {success_count} file vocal vao file ZIP: {zip_local}...')

with zipfile.ZipFile(zip_local, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
    for f in OUTPUT_VOCALS_DIR.glob('*_vocals.wav'):
        zf.write(f, f.name)

zip_size_mb = round(zip_local.stat().st_size / (1024**2), 2)
print(f'[OK] Da tao file ZIP ({zip_size_mb} MB)!')

print(f'[INFO] Dang copy file ZIP sang Google Drive: {DRIVE_OUTPUT_ZIP}...')
shutil.copyfile(zip_local, DRIVE_OUTPUT_ZIP)
print('[SUCCESS] Hoan tat! File vocal da san sang tren Google Drive de nạp vao Notebook Danh gia!')
